# Multi-Hinge Spine Mechanism Design

## Project Overview
This project generates the manufacturing files for a **multi-hinge spine mechanism** using a five-layer laminate process. The design consists of rigid panels connected by flexible castellated hinges, allowing for controlled bending and structural stability.

### Workflow
The code is structured into modular manufacturing steps:
1.  **Geometry Extraction:** Reads the input DXF (`spine.dxf`) to identify body, holes, and joint lines.
2.  **Hinge Generation:** Automatically generates castellated hinge patterns based on desired fold angles.
3.  **Laminate Assembly:** Stacks five layers (Structural-Adhesive-Flexure-Adhesive-Structural) and subtracts material.
4.  **Fabrication Prep:** Generates alignment jigs, weeding webs, and laser support structures.
5.  **Export:** Outputs `first_pass.dxf` (internal cuts) and `final_cut.dxf` (release cuts) into the `dxf/` folder.

In [8]:
import os
import foldable_robotics.dxf as frd
import foldable_robotics as fr
import foldable_robotics.manufacturing as frm
import foldable_robotics.parts.castellated_hinge2 as frc
from foldable_robotics.layer import Layer
from foldable_robotics.laminate import Laminate
import shapely.geometry as sg
import dxfv2 as dv

# ==========================================
# CONFIGURATION
# ==========================================
# All design parameters are centralized here for easy tuning.
CONFIG = {
    "input_file": "dxf/input/jensen-leg.dxf",  # Input CAD file
    "output_folder": "dxf/output",  # Output directory
    "thickness": 1,  # Material thickness (mm)
    "fold_angle": 120,  # Max fold angle for hinges
    "structure_angle": 90,  # Fold angle for structural joints
    "support_width": 2,  # Width of support web
    "kerf": 0.05,  # Laser cutter kerf
    "num_layers": 5,  # Total laminate layers
    "jig_diameter": 5,  # Alignment hole diameter
    "jig_spacing": 10,  # Grid spacing for jigs
    "is_adhesive": [False, True, False, True, False],  # Layer stackup
    "arc_approx": 10,  # Curve resolution
    "bridge_thickness": 1,  # Thickness of support bridges
    "bounding_box_padding": 10,  # Padding around the sheet
}

# Specific indices from the CAD file identifying joint types
HINGE_INDICES = [10, 11, 12, 13, 14, 15, 16, 25, 34]
STRUCTURE_INDICES = [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
]


In [9]:
def setup_environment():
    """Configures library resolution and ensures output directory exists."""
    fr.display_height = 300
    fr.resolution = 4
    if not os.path.exists(CONFIG["output_folder"]):
        os.makedirs(CONFIG["output_folder"])
    print(f"Environment set. Output folder '{CONFIG['output_folder']}' ready.")


setup_environment()


Environment set. Output folder 'dxf/output' ready.


## 1. Geometry Extraction
We parse the DXF file to separate the rigid body geometry from holes, cuts, and joint lines. This prepares clean Shapely polygons for the subsequent boolean operations.

In [10]:
def extract_body_geometry(filename):
    """Reads body, holes, and cuts to create the base Layer."""
    # Read Body
    body_vertices = dv.read_lwpolylines(
        filename, layer="body", arc_approx=CONFIG["arc_approx"]
    )
    body_polygons = [sg.Polygon(items).buffer(0) for items in body_vertices]
    body_layer = Layer(*body_polygons)

    # Read Holes
    hole_vertices = dv.read_lwpolylines(
        filename, layer="holes", arc_approx=CONFIG["arc_approx"]
    )
    circle_specs = dv.read_circles(filename, layer="holes")
    circle_polys = [
        sg.Point(center[0], center[1]).buffer(radius, quad_segs=16)
        for center, radius in circle_specs
    ]
    hole_layer = Layer(*[sg.Polygon(item) for item in hole_vertices + circle_polys])

    # Read Cuts
    cut_vertices = dv.read_lines(filename, layer="cuts")
    cut_layer = Layer(*[sg.LineString(item) for item in cut_vertices])
    cut_layer <<= 0.5  # Buffer cuts

    # Combine
    body_layer -= hole_layer
    body_layer -= cut_layer

    return body_layer


def extract_joints(filename, body_layer):
    """Reads joint lines and categorizes them into hinge and structural lines."""
    joint_vertices = dv.read_lines(filename, layer="joints")

    hinge_vertices = [joint_vertices[i] for i in HINGE_INDICES]
    structure_vertices = [joint_vertices[i] for i in STRUCTURE_INDICES]

    hinge_layer = Layer(*[sg.LineString(items) for items in hinge_vertices])
    structure_layer = Layer(*[sg.LineString(items) for items in structure_vertices])

    # Trim lines to stay inside the body
    hinge_layer &= body_layer
    structure_layer &= body_layer

    # Return coordinates for mapping
    mapped_hinges = [list(item.coords) for item in hinge_layer.geoms]
    mapped_structure = [list(item.coords) for item in structure_layer.geoms]

    return mapped_hinges, mapped_structure


## 2. Hinge Pattern Generation
Based on the input lines and desired fold angles, we generate castellated hinge patterns. These patterns are mapped onto the specific line segments identified in the previous step.

In [11]:
def generate_hinge_patterns(hinge_coords, struct_coords):
    """Generates castellated hinge geometry and maps it to the lines."""

    # Calculate widths based on thickness and desired angles
    w_deg, g_deg = frm.castellated_hinge_width(
        CONFIG["fold_angle"], CONFIG["thickness"]
    )
    w_str, g_str = frm.castellated_hinge_width(
        CONFIG["structure_angle"], CONFIG["thickness"]
    )

    # Generate patterns
    deg_pattern = frc.generate(g_deg, w_deg)
    str_pattern = frc.generate(g_str, w_str)

    # Map patterns to lines
    lam_deg = Layer().to_laminate(len(deg_pattern))
    lam_str = Layer().to_laminate(len(str_pattern))

    all_deg_hinges = []
    for p3, p4 in hinge_coords:
        all_deg_hinges.append(deg_pattern.map_line_stretch((0, 0), (1, 0), p3, p4))

    all_str_hinges = []
    for p3, p4 in struct_coords:
        all_str_hinges.append(str_pattern.map_line_stretch((0, 0), (1, 0), p3, p4))

    # Union all hinge geometries
    final_deg = lam_deg.unary_union(*all_deg_hinges)
    final_str = lam_str.unary_union(*all_str_hinges)

    return final_deg, final_str


## 3. Laminate Assembly & Web Generation
We assemble the full five-layer laminate, insert support bridges, and calculate the manufacturing web (scrap material) that holds the pieces in place during cutting.

In [12]:
def assemble_device(body_layer, hinge_geoms, struct_geoms, filename):
    """Stacks layers, subtracts hinges, and adds bridges."""

    # Create 5-layer stack
    device = Laminate(body_layer, body_layer, body_layer, body_layer, body_layer)
    device -= hinge_geoms
    device -= struct_geoms

    # Add Bridges
    bridges = dv.read_lines(filename, layer="bridge")
    bridges_layer = Layer(*[sg.LineString(item) for item in bridges])
    bridges_layer <<= CONFIG["bridge_thickness"]

    # Bridges on top/bottom structural layers
    bridges_lam = Laminate(
        bridges_layer, bridges_layer, Layer(), bridges_layer, bridges_layer
    )

    supported_device = device | bridges_lam

    # Clean up bridge artifacts
    diff = supported_device - device
    removal = frm.cleanup(diff, 0.1)
    removal = frm.keepout_laser(removal)

    final_device = device - removal
    return final_device


def generate_web_and_jigs(device):
    """Creates the manufacturing web, alignment holes, and ID tags."""

    # Bounding Box & Jig Holes
    (x1, y1), (x2, y2) = device.bounding_box_coords()
    w1, h1 = device.get_dimensions()

    w2 = (
        round(w1 / CONFIG["jig_spacing"]) * CONFIG["jig_spacing"]
        + CONFIG["jig_spacing"]
        + CONFIG["support_width"]
    )
    h2 = (
        round(h1 / CONFIG["jig_spacing"]) * CONFIG["jig_spacing"]
        + CONFIG["jig_spacing"]
        + CONFIG["support_width"]
    )

    x1 -= (w2 - w1) / 2
    y1 -= (h2 - h1) / 2
    x2 += (w2 - w1) / 2
    y2 += (h2 - h1) / 2

    points = [sg.Point(x1, y1), sg.Point(x2, y1), sg.Point(x1, y2), sg.Point(x2, y2)]
    holes_layer = Layer(*points)
    holes_layer <<= CONFIG["jig_diameter"] / 2
    alignment_holes = holes_layer.to_laminate(CONFIG["num_layers"])

    # Sheet Boundary
    sheet_layer = (holes_layer << CONFIG["bounding_box_padding"]).bounding_box()
    sheet = sheet_layer.to_laminate(CONFIG["num_layers"])

    # Layer IDs
    layer_id = frm.build_layer_numbers(
        CONFIG["num_layers"], text_size=CONFIG["jig_diameter"]
    )
    layer_id = layer_id.simplify(0.2)

    # Web Calculation
    removable = frm.calculate_removable_scrap(
        device, sheet, CONFIG["support_width"], CONFIG["is_adhesive"]
    )
    web = (
        removable
        - alignment_holes
        - layer_id.translate(
            x1 + CONFIG["jig_diameter"], y1 - CONFIG["jig_diameter"] / 2
        )
    )

    return web, sheet, alignment_holes


## 4. Export Manufacturing Files
Finally, we separate the design into two cut files:
1.  **First Pass:** Internal cuts, hinges, and alignment holes.
2.  **Final Cut:** The outer boundary that releases the device from the sheet.

In [13]:
def prepare_and_export(device, web, sheet):
    """Calculates supports, keeps out regions, and exports DXF files."""

    # Laser Keepout
    keepout = frm.keepout_laser(device)

    # Supports
    support = frm.support(
        device, frm.keepout_laser, CONFIG["support_width"], CONFIG["support_width"] / 2
    )
    supported_design = web | device | support

    # --- 1. Export First Pass (Internal Cuts) ---
    w, h = supported_design.get_dimensions()
    p0, p1 = supported_design.bounding_box_coords()

    # Arrange Rigid Layers (Top and Bottom)
    rigid_layer = supported_design[0] | (supported_design[-1].translate(w + 5, 0))

    # Arrange Adhesive Layers (Middle layers) - Note the mirroring logic for layer 3
    l4 = supported_design[3].scale(-1, 1)  # Mirror top adhesive
    p2, p3 = l4.bounding_box_coords()
    l4 = l4.translate(
        p0[0] - p2[0] + w + 5, p0[1] - p2[1]
    )  # Move next to bottom adhesive
    adhesive_layer = supported_design[1] | l4

    # Assemble First Pass Laminate
    first_pass = Laminate(rigid_layer, adhesive_layer, supported_design[2])

    fp_path = os.path.join(CONFIG["output_folder"], "first_pass")
    first_pass.export_dxf(fp_path)
    print(f"Exported: {fp_path}.dxf")

    # --- 2. Export Final Cut (Outer Profile) ---
    final_cut = sheet - keepout
    final_cut = final_cut[0]  # Select one layer profile

    fc_path = os.path.join(CONFIG["output_folder"], "final_cut")
    final_cut.export_dxf(fc_path)
    print(f"Exported: {fc_path}.dxf")


In [14]:
# ==========================================
# MAIN EXECUTION
# ==========================================

print("Processing Geometry...")
body = extract_body_geometry(CONFIG["input_file"])
hinge_lines, struct_lines = extract_joints(CONFIG["input_file"], body)

print("Generating Hinges...")
hinge_geoms, struct_geoms = generate_hinge_patterns(hinge_lines, struct_lines)

print("Assembling Laminate...")
final_device = assemble_device(body, hinge_geoms, struct_geoms, CONFIG["input_file"])

print("Generating Web and Supports...")
web, sheet, holes = generate_web_and_jigs(final_device)

print("Exporting Manufacturing Files...")
prepare_and_export(final_device, web, sheet)

print("Done. Check the 'dxf' folder for output files.")


Processing Geometry...
Generating Hinges...
Assembling Laminate...
Generating Web and Supports...
Exporting Manufacturing Files...
Exported: dxf/output/first_pass.dxf
Exported: dxf/output/final_cut.dxf
Done. Check the 'dxf' folder for output files.
